## Import Python Libraries and load environment variables

In [1]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

/workspaces/hai5016-project/.venv/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [3]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [4]:
# bfmsgs(result)

## Create a dummy function that returns 1500 as KRW > USD FX

In [5]:
def get_fx(currency: str) -> float:
    """Return conversion rate from KRW to the given currency (e.g., USD)."""
    # Read API key from environment variables loaded by load_dotenv()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Build ExchangeRate-API endpoint with KRW as base currency
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/KRW"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    # Optional API-level error check
    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code])

In [6]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [7]:
from datetime import datetime
import re

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch menu-like text blocks from a URL and keep only useful content."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as error:
        return f"Fetch failed: {error}"

    # Decode HTML safely
    html = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    # Remove non-content tags that add a lot of noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()

    # Keep likely menu containers first (table/list/sections with diet/menu keywords)
    candidate_blocks = []
    keyword_pattern = re.compile(r"(식단|메뉴|중식|석식|조식|lunch|dinner|breakfast|menu)", re.IGNORECASE)

    selectors = [
        "table",
        "ul",
        "ol",
        "section",
        "article",
        "div",
    ]

    for selector in selectors:
        for node in soup.select(selector):
            text = node.get_text(" ", strip=True)
            if len(text) < 30:
                continue
            if keyword_pattern.search(text):
                candidate_blocks.append(text)

    # Fallback to body text if no candidate block is found
    if not candidate_blocks and soup.body:
        candidate_blocks = [soup.body.get_text(" ", strip=True)]

    # De-duplicate and keep output small enough for the model
    unique_blocks = []
    seen = set()
    for block in candidate_blocks:
        normalized = " ".join(block.split())
        if normalized not in seen:
            seen.add(normalized)
            unique_blocks.append(normalized)

    # Include today's date to improve downstream filtering
    today = datetime.now()
    today_hint = (
        f"TODAY_HINT: {today.strftime('%Y-%m-%d')} / "
        f"{today.strftime('%m/%d')} / "
        f"{today.strftime('%-m/%-d')}"
    )

    trimmed_text = "\n\n".join(unique_blocks[:20])
    return f"SOURCE_URL: {url}\n{today_hint}\n\n{trimmed_text[:25000]}"

In [8]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [9]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [10]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12"}]}
)
bfmsgs(result)

Here’s today’s menu from Hanyang University’s campuses (as of 2026-05-21 on the site you gave).

Seoul campus
- Breakfast (조식, 천원의아침밥): Braised Pork with Rice Cake in Soy Sauce (간장돈육떡장조림) with rice, kimchi-jeon, lunchbox seaweed, tri-color mustard salad, kkakdugi. Price: 1,000원
- Lunch (중식): Hayashi Rice with Sausage (소시지하이라이스) with chives salad, kimchi, miso-style soup. Price: 4,500원
- Main option: Spicy Stir-fried Pork (돼지고기고추장볶음) with rice, ham with ketchup, cabbage wraps, ssamjang, chives salad, kimchi. Price: 6,500원

ERICA campus
- Breakfast (조식, 천원의아침밥): Braised Pork with Rice Cake in Soy Sauce (간장돈육떡장조림) with rice, kimchi-jeon, lunchbox seaweed, tri-color mustard salad, kkakdugi. Price: 1,000원
- Lunch (중식): Hayashi Rice with Sausage (소시지하이라이스) with chives salad, kimchi, miso-style soup. Price: 4,500원
- Main option: Spicy Stir-fried Pork (돼지고기고추장볶음) with rice, ham with ketchup, cabbage wraps, ssamjang, chives salad, kimchi. Price: 6,500원

Notes
- Menu items and prices come from the “오늘의 메뉴” section of the site; items can change.
- Prices are in KRW.

Would you like me to convert these prices to USD or another currency?